---
Methods Project Part 1
---


In [6]:
#Download and unzip Dataset from kaggle as uci archive returns 502 bad gateway
#https://www.kaggle.com/datasets/uciml/breast-cancer-wisconsin-data
!curl -L -o breast-cancer-wisconsin-data.zip\
  https://www.kaggle.com/api/v1/datasets/download/uciml/breast-cancer-wisconsin-data
!unzip breast-cancer-wisconsin-data.zip

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 49796  100 49796    0     0   185k      0 --:--:-- --:--:-- --:--:--  185k
Archive:  breast-cancer-wisconsin-data.zip
  inflating: data.csv                


## Part 1 - Data Preprocessing

### 1. Data cleaning & missing value prediction
In this section, I run logistic regression on noisy version of dataset with two noise levels ($p=20\%$ and $p=40\%$). First case is creating noise on feature 'radius_mean', second one is creating noise in random feature. We then compare the model's performance on these noisy datasets before imputation (by dropping rows with NaNs) and after applying Scikit-Learn's **SimpleImputer** and **IterativeImputer**. To prevent data leakage, imputers are strictly fitted on the training folds and applied to the test folds during the Stratified 5-Fold Cross-Validation.

In [19]:
#Importing all the required library and classes
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer,SimpleImputer
from typing import Union,Optional
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score
from sklearn.decomposition import TruncatedSVD 
from sklearn.neural_network import MLPRegressor

#Global Variables for consistency
#For reproducible results across model executions
RANDOM_STATE = 1 #Seed or randomness for all the algorithms including kfold cv
TOTAL_ITERATIONS = 1000 #Fixed number of number for all classes
K_FOLD = 5 #Fixed K for CV

#Load the data
bcw_df = pd.read_csv('data.csv')
#Overview through first 5 rows
bcw_df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [20]:
#Column Id and Unnamed: 32 are not useful, One is just Ids for reference and other appears to only contain NaNs
if 'Unnamed: 32' in bcw_df.columns and 'id' in bcw_df.columns:
    bcw_df.drop(columns=['id','Unnamed: 32'],inplace=True)

#Separate target and features before adding noise to the data
X = bcw_df.drop(columns=['diagnosis'])
y = bcw_df['diagnosis'].map(lambda value: 1 if value == 'M' else 0 )
dimension = X.shape[1]

This generic function executes a Logistic Regression model using Stratified 5-Fold Cross-Validation. It is designed to handle missing values (NaNs) based on the provided arguments: it applies either IterativeImputer or SimpleImputer if specified; otherwise, it defaults to dropping the rows containing NaNs.

Rows are dropped rather than filled with zeros or arbitrary constants because such replacements are themselves a form of imputation that can introduce bias. Since Scikit-Learn’s Logistic Regression implementation cannot natively process datasets containing missing values, dropping the affected rows serves as the strictly non-imputed baseline for performance comparison.

In [21]:
def run_stratified_k_fold_logistic_regression(X_data:pd.DataFrame,imputer: Optional[Union[IterativeImputer,SimpleImputer]] = None):
    """ Takes Dataframe to run logistic regression using stratified 5-fold CV
        Performs imputation if provided on test and training data separately to avoid leakage"""
    
    straified_5_fold = StratifiedKFold(n_splits=K_FOLD,shuffle=True,random_state=RANDOM_STATE)
    #Track F1 Scores across 5 folds
    f1_scores = []
    
    for train_idx,test_idx in straified_5_fold.split(X_data,y):
        # Separate train and test data
        X_train, X_test = X_data.loc[train_idx], X_data.loc[test_idx]
        y_train, y_test = y.loc[train_idx], y.loc[test_idx]
        
        #Impute missing values (if imputer is provided)
        if imputer:
            #Fit on train, transform BOTH train and test
            X_train = imputer.fit_transform(X_train)
            X_test = imputer.transform(X_test)
        else:
        #Else drop NaNs rows as SkLearn logistic regression natively cannot handle NaNs 
            train_non_nan_mask = ~np.isnan(X_train).any(axis=1)
            X_train,y_train = X_train[train_non_nan_mask], y_train[train_non_nan_mask]
            test_non_nan_mask = ~np.isnan(X_test).any(axis=1)
            X_test,y_test = X_test[test_non_nan_mask], y_test[test_non_nan_mask]
            
        #Using Z score standardization for train and test data
        #leaving unstandardized will make the model convergence difficult and will bias the output
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)
            
        logit_model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
        logit_model.fit(X_train,y_train)
        logit_model_predictions = logit_model.predict(X_test)
        #Calculate F1 score on testing data
        model_f1_score = f1_score(y_true=y_test,y_pred=logit_model_predictions)
        f1_scores.append(model_f1_score)
        
    #Return mean and standard deviation of f1_scores for the model for future evaluation     
    return np.mean(f1_scores),np.std(f1_scores)


In [22]:

#P values or noise levels to test for
p_values = [0.2,0.4]
for p in p_values:
    #Get random p% idx of total idx 
    radius_noise_p_idx = bcw_df.sample(frac=p).index
    random_noise_p_idx = bcw_df.sample(frac=p).index
    
    #Clonning the dataframe
    X_radius_noise = X.copy()
    X_random_noise = X.copy()

    #Generate noise
    #Single feature
    X_radius_noise.loc[radius_noise_p_idx,'radius_mean'] = np.nan
    #Random feature
    for idx in random_noise_p_idx:
        random_feature = np.random.choice(X_random_noise.columns)
        X_random_noise.loc[idx,random_feature] = np.nan
    
    #Logistic regression on p% noise for radius feature without imputer
    mean_f1_radius_noise_no_imputer,std_f1_radius_noise_no_imputer = run_stratified_k_fold_logistic_regression(X_radius_noise)
    
    #Logistic regression on p% noise for radius feature with simple imputer
    mean_f1_radius_noise_simple_imputer,std_f1_radius_noise_simple_imputer = run_stratified_k_fold_logistic_regression(X_radius_noise,SimpleImputer())
    
    #Logistic regression on p% noise for radius feature with Iterative imputer
    mean_f1_radius_noise_iterative_imputer,std_f1_radius_noise_iterative_imputer = run_stratified_k_fold_logistic_regression(X_radius_noise,IterativeImputer())
    
    #Logistic regression on p% noise for random feature without imputer
    mean_f1_random_noise_no_imputer,std_f1_random_noise_no_imputer = run_stratified_k_fold_logistic_regression(X_random_noise)
    
    #Logistic regression on p% noise for random feature with simple imputer
    mean_f1_random_noise_simple_imputer,std_f1_random_noise_simple_imputer = run_stratified_k_fold_logistic_regression(X_random_noise,SimpleImputer())
    
    #Logistic regression on p% noise for random feature with iterative imputer
    mean_f1_random_noise_iterative_imputer,std_f1_random_noise_iterative_imputer = run_stratified_k_fold_logistic_regression(X_random_noise,IterativeImputer())
    
    print('---')
    print(f'Results for p = {int(p*100)}%')
    print(f'No Imputer        | Single Feat: {mean_f1_radius_noise_no_imputer:.4f} +/- {std_f1_radius_noise_no_imputer:.4f} | Random Feat: {mean_f1_random_noise_no_imputer:.4f} +/- {std_f1_random_noise_no_imputer:.4f}')
    print(f'Simple Imputer    | Single Feat: {mean_f1_radius_noise_simple_imputer:.4f} +/- {std_f1_radius_noise_simple_imputer:.4f} | Random Feat: {mean_f1_random_noise_simple_imputer:.4f} +/- {std_f1_random_noise_simple_imputer:.4f}')
    print(f'Iterative Imputer | Single Feat: {mean_f1_radius_noise_iterative_imputer:.4f} +/- {std_f1_radius_noise_iterative_imputer:.4f} | Random Feat: {mean_f1_random_noise_iterative_imputer:.4f} +/- {std_f1_random_noise_iterative_imputer:.4f}')    
    print('---')
    
    #Plotly bar charts with error bars
    fig = go.Figure()
    fig.add_trace(go.Bar(
    name='Radius feature noise',
    x=['No Imputer', 'Simple Imputer', 'Iterative Imputer'], y=[mean_f1_radius_noise_no_imputer, mean_f1_radius_noise_simple_imputer,mean_f1_radius_noise_iterative_imputer],
    error_y=dict(type='data', array=[std_f1_radius_noise_no_imputer, std_f1_radius_noise_simple_imputer, std_f1_radius_noise_iterative_imputer])
    ))
    fig.add_trace(go.Bar(
        name='Random feature noise',
        x=['No Imputer', 'Simple Imputer', 'Iterative Imputer'], y=[mean_f1_random_noise_no_imputer, mean_f1_random_noise_simple_imputer, mean_f1_random_noise_iterative_imputer],
        error_y=dict(type='data', array=[std_f1_random_noise_no_imputer, std_f1_random_noise_simple_imputer, std_f1_random_noise_iterative_imputer])
    ))
    fig.update_layout(barmode='group',title=f'F1 Scores for p={p*100}% Missing Data',
        xaxis_title='Imputation Method',
        yaxis_title='Mean F1 Score',
        width=1000, 
        height=600)
    fig.show()
    
    

---
Results for p = 20%
No Imputer        | Single Feat: 0.9719 +/- 0.0188 | Random Feat: 0.9619 +/- 0.0147
Simple Imputer    | Single Feat: 0.9712 +/- 0.0128 | Random Feat: 0.9639 +/- 0.0178
Iterative Imputer | Single Feat: 0.9709 +/- 0.0188 | Random Feat: 0.9709 +/- 0.0188
---


---
Results for p = 40%
No Imputer        | Single Feat: 0.9492 +/- 0.0204 | Random Feat: 0.9593 +/- 0.0140
Simple Imputer    | Single Feat: 0.9709 +/- 0.0188 | Random Feat: 0.9689 +/- 0.0124
Iterative Imputer | Single Feat: 0.9709 +/- 0.0188 | Random Feat: 0.9711 +/- 0.0127
---


### 2. Dimensionality reduction

In this section, I compare two dimensionality reduction techniques: linear (Truncated SVD) and non-linear (Autoencoder). I use **TruncatedSVD** to reduce the feature space to $k$ dimensions.
For Autoencoder I configure an **MLPRegressor** to predict the input data. The network has an hourglass architecture: input $\rightarrow$ min(d, 2k) $\rightarrow$ k (latent bottleneck) $\rightarrow$ min(d, 2k) $\rightarrow$ output.

In [23]:
#We manually extract the learned weights W1,W2 and biases b1,b2 from the trained MLP 
#to compute the k-dimensional latent representation of the data.
def get_latent_ae(X_input, ae_model):
    """Extracts the latent representation (k-dimensional) from the Autoencoder."""
    layer0 = X_input
    
    # W1/b1 are weights/biases for the first hidden layer, W2/b2 are for the bottleneck (latent) layer
    W1, W2 = ae_model.coefs_[0], ae_model.coefs_[1]
    b1, b2 = ae_model.intercepts_[0], ae_model.intercepts_[1]
    
    A1 = np.maximum(0, np.dot(layer0, W1) + b1) # ReLU activation for layer 1
    A2 = np.maximum(0, np.dot(A1, W2) + b2)     # ReLU activation for latent layer (k)
    return A2

#Run logistic regression without any dimensionaltiy reduction or noise
mean_f1_base,std_f1_base = run_stratified_k_fold_logistic_regression(X)
print(f'F1 score with no reduction/noise: {mean_f1_base:.4f} +/- {std_f1_base:.4f}\n')

#Keep track of f1 mean and std across K values run
mean_f1_svd , mean_f1_ae = [],[]
std_f1_svd,std_f1_ae = [], []
k_values = [2,5,10]
for k in k_values:
    straified_5_fold = StratifiedKFold(n_splits=K_FOLD,shuffle=True,random_state=RANDOM_STATE)
    #Track F1 Scores across 5 folds
    f1_logit_svd,f1_logit_ae = [],[]
    
    for train_idx,test_idx in straified_5_fold.split(X,y):
        # Separate train and test data
        X_train, X_test = X.loc[train_idx], X.loc[test_idx]
        y_train, y_test = y.loc[train_idx], y.loc[test_idx]
        
        #Using Z score standardization for train and test data
        #leaving unstandardized will make the model convergence difficult and will bias the output
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)
        
        #SVD
        svd = TruncatedSVD(n_components=k,n_iter=TOTAL_ITERATIONS,random_state=RANDOM_STATE)
        X_train_svd = svd.fit_transform(X_train)
        X_test_svd = svd.transform(X_test)
        
        #Using svd reduced data to logistic regression
        logit_svd = LogisticRegression(max_iter = TOTAL_ITERATIONS,random_state = RANDOM_STATE)
        logit_svd.fit(X_train_svd,y_train)
        logit_svd_predictions = logit_svd.predict(X_test_svd)
        f1_logit_svd.append(f1_score(y_true=y_test,y_pred=logit_svd_predictions))

        #Autoencoder
        w1 = min(dimension,2*k)
        
        #Setting up Autoencoder with layers: input -> min(d, 2*k) -> k (latent) -> min(d, 2*k) -> output
        ae = MLPRegressor(hidden_layer_sizes=(w1,k,w1),activation='relu',random_state=RANDOM_STATE,max_iter=TOTAL_ITERATIONS)
        
        #Autoencoder goal is not to predict target label but it is to reconstruct original input data at the output layer
        ae.fit(X_train,X_train)
        X_train_ae = get_latent_ae(X_train, ae)
        X_test_ae = get_latent_ae(X_test, ae)
        #Logistic regression for autoencoder data
        logit_ae = LogisticRegression(max_iter = TOTAL_ITERATIONS,random_state = RANDOM_STATE)
        logit_ae.fit(X_train_ae, y_train)
        logit_ae_predictions = logit_ae.predict(X_test_ae)
        f1_logit_ae.append(f1_score(y_true=y_test,y_pred= logit_ae_predictions))
        
    #Store mean and std of F1 for kfold CV run and print    
    mean_f1_svd.append(np.mean(f1_logit_svd))
    std_f1_svd.append(np.std(f1_logit_svd))
    mean_f1_ae.append(np.mean(f1_logit_ae))
    std_f1_ae.append(np.std(f1_logit_ae))
    print(f'k={k:2d} | SVD: {np.mean(f1_logit_svd):.4f} +/- {np.std(f1_logit_svd):.4f} | AE: {np.mean(f1_logit_ae):.4f} +/- {np.std(f1_logit_ae):.4f}')

#plotting the findings using plotly
fig = go.Figure()
fig.add_trace(go.Scatter(
        name= 'SVD',
        x=k_values,
        y=mean_f1_svd,
        error_y=dict(type='data', array=std_f1_svd)
        ))
fig.add_trace(go.Scatter(
        name= 'Autoencoder',
        x=k_values,
        y=mean_f1_ae,
        error_y=dict(type='data', array=std_f1_ae)
    ))
fig.add_trace(go.Scatter(
    name='base logistic regression',
    x=[min(k_values), max(k_values)], 
    y=[mean_f1_base, mean_f1_base],  
    mode='lines',                     
    line=dict(dash='dash', color='red')
))
fig.update_layout(title=f'Mean F1 Scores for logistic regression using SVD and AE',
        xaxis_title='Dimensionality of latent representation',
        yaxis_title='Mean F1 Score',
        width=1000, 
        height=600)
fig.show()
        

F1 score with no reduction/noise: 0.9709 +/- 0.0188

k= 2 | SVD: 0.9381 +/- 0.0117 | AE: 0.9169 +/- 0.0247
k= 5 | SVD: 0.9569 +/- 0.0169 | AE: 0.9432 +/- 0.0145
k=10 | SVD: 0.9737 +/- 0.0160 | AE: 0.9661 +/- 0.0142
